# 2주차 배포 노트북: 식품관, 패션관 테스트 API

위에서부터 순서대로 실행한다. 콘솔 작업 없이 `infra/stack.yaml` 스택 하나로 모든 리소스를 만들고 지운다.

**시작 전:** 왼쪽 열쇠 아이콘(Secrets)에 `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`를 넣고 노트북 접근을 켠다. 키를 셀에 직접 적지 않는다.

**과금:** 3번 셀부터 ALB, 인터페이스 엔드포인트, Fargate 작업 요금이 시간 단위로 나간다. 확인이 끝나면 마지막 셀로 전부 지운다.

## 1. 준비

In [ ]:
!pip -q install boto3

In [ ]:
import io, json, time, zipfile, urllib.request, urllib.error
import boto3, botocore
from google.colab import userdata

REGION = "ap-northeast-2"
STACK = "ragbot-week2"
REPO = "ChangSik88/CloudArchitecture"
BRANCH = "main"  # PR 머지 전이면 기능 브랜치 이름으로 바꾼다

session = boto3.Session(
    aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY"),
    region_name=REGION,
)
cfn = session.client("cloudformation")
cb = session.client("codebuild")
ecs = session.client("ecs")
s3 = session.client("s3")
ddb = session.client("dynamodb")
aas = session.client("application-autoscaling")
print("계정:", session.client("sts").get_caller_identity()["Arn"])

## 2. GitHub 소스 받기

템플릿과 빌드용 소스를 저장소에서 받는다. zip의 최상위 폴더를 벗겨 CodeBuild가 `buildspec.yml`을 바로 찾게 한다.

In [ ]:
url = f"https://codeload.github.com/{REPO}/zip/refs/heads/{BRANCH}"
raw = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
top = raw.namelist()[0].split("/")[0] + "/"

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    for name in raw.namelist():
        if name.startswith(top) and not name.endswith("/"):
            z.writestr(name[len(top):], raw.read(name))
SOURCE_ZIP = buf.getvalue()
TEMPLATE = raw.read(top + "infra/stack.yaml").decode("utf-8")
print(f"소스 {len(SOURCE_ZIP):,} bytes, 템플릿 {len(TEMPLATE):,} chars")

## 3. 스택 배포 함수

In [ ]:
def outputs():
    s = cfn.describe_stacks(StackName=STACK)["Stacks"][0]
    return {o["OutputKey"]: o["OutputValue"] for o in s.get("Outputs", [])}

def show_failures():
    for e in cfn.describe_stack_events(StackName=STACK)["StackEvents"][:30]:
        if "FAILED" in e["ResourceStatus"]:
            print(e["LogicalResourceId"], e["ResourceStatus"], e.get("ResourceStatusReason", ""))

def deploy(desired):
    args = dict(
        StackName=STACK,
        TemplateBody=TEMPLATE,
        Parameters=[{"ParameterKey": "DesiredCount", "ParameterValue": str(desired)}],
        Capabilities=["CAPABILITY_IAM"],
    )
    try:
        cfn.describe_stacks(StackName=STACK)
        exists = True
    except botocore.exceptions.ClientError:
        exists = False
    try:
        if exists:
            cfn.update_stack(**args)
            waiter = "stack_update_complete"
        else:
            cfn.create_stack(**args)
            waiter = "stack_create_complete"
    except botocore.exceptions.ClientError as e:
        if "No updates are to be performed" in str(e):
            print("변경 없음")
            return outputs()
        raise
    print(waiter, "대기 중 (처음 만들 때 5~10분)")
    try:
        cfn.get_waiter(waiter).wait(StackName=STACK, WaiterConfig={"Delay": 15, "MaxAttempts": 160})
    except botocore.exceptions.WaiterError:
        show_failures()
        raise
    print("완료")
    return outputs()

## 4. 1단계: 작업 0개로 스택 생성

이미지가 아직 없으므로 ECS 서비스는 작업 0개로 만든다.

In [ ]:
out = deploy(0)
out

## 5. 이미지 빌드 (CodeBuild)

소스를 S3 `build/`에 올리면 CodeBuild가 AWS 안에서 Docker 이미지를 빌드해 ECR에 올린다.

In [ ]:
s3.put_object(Bucket=out["DocsBucket"], Key="build/source.zip", Body=SOURCE_ZIP)
build_id = cb.start_build(projectName=out["BuildProject"])["build"]["id"]
while True:
    b = cb.batch_get_builds(ids=[build_id])["builds"][0]
    print(b["currentPhase"], b["buildStatus"])
    if b["buildStatus"] != "IN_PROGRESS":
        break
    time.sleep(15)
assert b["buildStatus"] == "SUCCEEDED", b.get("logs", {}).get("deepLink")
print("이미지:", out["ImageRepoUri"] + ":latest")

## 6. 2단계: 서비스별 작업 2개 실행

In [ ]:
out = deploy(2)
CLUSTER = out["Cluster"]
SERVICES = {"food": out["FoodService"], "fashion": out["FashionService"]}
for svc in SERVICES.values():  # 이미지를 다시 빌드했을 때 새 이미지로 바꾼다
    ecs.update_service(cluster=CLUSTER, service=svc, forceNewDeployment=True)
print("서비스 안정화 대기 중 (3~5분)")
ecs.get_waiter("services_stable").wait(cluster=CLUSTER, services=list(SERVICES.values()))
BASE = "http://" + out["AlbDns"]
print("ALB 주소:", BASE)

## 7. 확인 1: 경로 분기와 가용 영역 분산

같은 ALB 주소인데 경로에 따라 다른 서비스가 응답하고, 요청이 두 가용 영역의 작업에 나뉘어 간다.

In [ ]:
def call(path, method="GET", body=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(BASE + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()[:200]

for svc in SERVICES:
    print(f"GET /api/{svc}/health ->", call(f"/api/{svc}/health"))
    for _ in range(4):
        print(f"    /api/{svc}/info   ->", call(f"/api/{svc}/info"))
print("GET /api/other ->", call("/api/other"))

## 8. 확인 2: 서비스별 대화 기록 쓰기

각 작업 역할은 자기 테이블에 쓰기만 할 수 있다. 조회는 노트북의 관리자 키로 한다.

In [ ]:
print(call("/api/food/log", "POST", {"message": "식품관 테스트 대화"}))
print(call("/api/fashion/log", "POST", {"message": "패션관 테스트 대화"}))
for t in [out["FoodTable"], out["FashionTable"]]:
    print(t, ddb.scan(TableName=t, Select="COUNT")["Count"], "건")

## 9. 확인 3: 장애 격리

식품관 작업을 0개로 줄여도 패션관은 계속 응답해야 한다. 테스트 동안만 식품관 오토스케일링을 멈춘다.

In [ ]:
FOOD = SERVICES["food"]
dim = dict(ServiceNamespace="ecs", ResourceId=f"service/{CLUSTER}/{FOOD}",
           ScalableDimension="ecs:service:DesiredCount")
aas.register_scalable_target(**dim, MinCapacity=0, SuspendedState={
    "DynamicScalingInSuspended": True, "DynamicScalingOutSuspended": True, "ScheduledScalingSuspended": True})
ecs.update_service(cluster=CLUSTER, service=FOOD, desiredCount=0)
while ecs.describe_services(cluster=CLUSTER, services=[FOOD])["services"][0]["runningCount"] > 0:
    time.sleep(10)
time.sleep(15)
print("식품관 작업 0개")
print("GET /api/food/health    ->", call("/api/food/health"))
print("GET /api/fashion/health ->", call("/api/fashion/health"))

In [ ]:
# 식품관 복구
ecs.update_service(cluster=CLUSTER, service=FOOD, desiredCount=2)
aas.register_scalable_target(**dim, MinCapacity=2, SuspendedState={
    "DynamicScalingInSuspended": False, "DynamicScalingOutSuspended": False, "ScheduledScalingSuspended": False})
ecs.get_waiter("services_stable").wait(cluster=CLUSTER, services=[FOOD])
print("GET /api/food/health ->", call("/api/food/health"))

## 10. 정리 (과금 중단)

버킷의 모든 버전을 지운 뒤 스택을 삭제한다. 되돌릴 수 없으므로 캡처를 다 받은 뒤 실행한다.

In [ ]:
out = outputs()
bucket = out["DocsBucket"]
for page in s3.get_paginator("list_object_versions").paginate(Bucket=bucket):
    objs = [{"Key": v["Key"], "VersionId": v["VersionId"]}
            for v in page.get("Versions", []) + page.get("DeleteMarkers", [])]
    if objs:
        s3.delete_objects(Bucket=bucket, Delete={"Objects": objs})
cfn.delete_stack(StackName=STACK)
print("삭제 대기 중")
cfn.get_waiter("stack_delete_complete").wait(StackName=STACK, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
print("삭제 완료")